# 02 – Első adat-migráció: vízállás és vízhőmérséklet

## A notebook célja

Ebben a lépésben végrehajtjuk az eredeti elemzési terv első adat-migrációját.

A 2016–2026 közötti CSV-fájlokat egy közös, egységes szerkezetű adatállományba olvassuk be. Ezután külön előállítjuk azt az adatállományt, amelyben a dátum, a vízállás és a vízhőmérséklet egyaránt érvényes.

**A szűrés szabálya:** csak az a rekord kerül az első elemzésre szánt adatállományba, amelynél mindhárom vizsgált mező értelmezhető:
- `Időpont`;
- `Siófok [2888] Felszíni vízállás (cm)`;
- `Siófok [2888] Vízhő a vízfelszín közelében (C)`.

A közös, szűrés előtti állományt is elmentjük, így a szűrés hatása később ellenőrizhető.

**Fontos:** az eredeti CSV-fájlokat nem módosítjuk. A notebook a projekt `data/interim` és `data/processed` könyvtáraiba készít új állományokat.


## 1. Könyvtárak és oszlopnevek beállítása

A nyers adatok helye a projekt gyökérkönyvtárán belül található. Ha a projektet másik gépre vagy másik könyvtárba helyezed, a `PROJECT_DIR` értékét kell módosítani.


In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np

PROJECT_DIR = Path(
    r"C:\Users\Zoli\Adatelemzéseim\Balatoni_vízállás_2016-2026_GIT"
)
RAW_DIR = PROJECT_DIR / "Raw_data_2016-2026"
INTERIM_DIR = PROJECT_DIR / "data" / "interim"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DATETIME_COL = "Időpont"
WATER_LEVEL_COL = "Siófok [2888]\n Felszíni vízállás (cm)"
WATER_TEMP_COL = "Siófok [2888]\n Vízhő a vízfelszín közelében (C)"

print("Projekt:", PROJECT_DIR)
print("Nyers adatok:", RAW_DIR)
print("Köztes állományok:", INTERIM_DIR)
print("Feldolgozott állományok:", PROCESSED_DIR)

if not RAW_DIR.exists():
    raise FileNotFoundError(f"Nem található a nyers adatkönyvtár: {RAW_DIR}")


Projekt: C:\Users\Zoli\Adatelemzéseim\Balatoni_vízállás_2016-2026_GIT
Nyers adatok: C:\Users\Zoli\Adatelemzéseim\Balatoni_vízállás_2016-2026_GIT\Raw_data_2016-2026
Köztes állományok: C:\Users\Zoli\Adatelemzéseim\Balatoni_vízállás_2016-2026_GIT\data\interim
Feldolgozott állományok: C:\Users\Zoli\Adatelemzéseim\Balatoni_vízállás_2016-2026_GIT\data\processed


## 2. A forrásfájlok és az oszlopnevek ellenőrzése

A fájlokat a `YYYY-YY.csv` névminta alapján keressük meg. A migráció csak akkor indul el, ha a három elvárt oszlop mindegyike pontosan egyszer megtalálható minden fájlban.


In [2]:
filename_pattern = re.compile(r"^\d{4}-\d{2}\.csv$", flags=re.IGNORECASE)
csv_files = sorted(
    p for p in RAW_DIR.glob("*.csv")
    if filename_pattern.match(p.name)
)

if not csv_files:
    raise FileNotFoundError(
        "Nem található a YYYY-YY.csv mintának megfelelő CSV-fájl."
    )

print(f"Feldolgozandó CSV-fájlok száma: {len(csv_files)}")
for path in csv_files:
    print("-", path.name)


Feldolgozandó CSV-fájlok száma: 10
- 2016-17.csv
- 2017-18.csv
- 2018-19.csv
- 2019-20.csv
- 2020-21.csv
- 2021-22.csv
- 2022-23.csv
- 2023-24.csv
- 2024-25.csv
- 2025-26.csv


## 3. CSV-beolvasó és adattípus-átalakító függvények

A korábbi feltárás alapján a fájlok jellemzően pontosvesszővel tagoltak és UTF-8 BOM kódolásúak. A beolvasó ennek ellenére több gyakori kódolást és elválasztót is megpróbál, és a használt beállításokat rögzíti.

A vízhőmérséklet szöveges (`object`) oszlopként érkezik. Az átalakítás a tizedesvesszőt tizedesponttá alakítja, majd numerikus értékké konvertálja. Az üres vagy nem értelmezhető értékek hiányzó értékként maradnak.


In [3]:
ENCODINGS = ["utf-8-sig", "cp1250", "latin2", "utf-8"]
SEPARATORS = [";", ",", "\t"]

def read_source_csv(file_path):
    errors = []
    for encoding in ENCODINGS:
        for separator in SEPARATORS:
            try:
                frame = pd.read_csv(
                    file_path,
                    sep=separator,
                    encoding=encoding,
                    engine="python"
                )
                if frame.shape[1] >= 3:
                    return frame, encoding, separator
                errors.append(
                    f"{encoding}, {repr(separator)}: {frame.shape[1]} oszlop"
                )
            except Exception as exc:
                errors.append(
                    f"{encoding}, {repr(separator)}: "
                    f"{type(exc).__name__}: {exc}"
                )
    raise ValueError(
        f"Nem sikerült beolvasni a fájlt: {file_path.name}\n"
        + "\n".join(errors)
    )


def parse_datetime(series):
    """Dátum-idő átalakítás; a nem értelmezhető érték NaT lesz."""
    text = series.astype("string").str.strip()
    try:
        return pd.to_datetime(
            text,
            errors="coerce",
            dayfirst=True,
            format="mixed"
        )
    except TypeError:
        # Régebbi pandas-verzióhoz tartalék megoldás.
        return pd.to_datetime(text, errors="coerce", dayfirst=True)


def parse_numeric(series):
    """Numerikus átalakítás tizedesvessző támogatásával."""
    text = series.astype("string").str.strip()
    text = text.replace({
        "": pd.NA,
        "-": pd.NA,
        "—": pd.NA,
        "–": pd.NA,
        "n.a.": pd.NA,
        "N/A": pd.NA,
        "None": pd.NA
    })
    text = text.str.replace(",", ".", regex=False)
    return pd.to_numeric(text, errors="coerce")


## 4. A CSV-fájlok beolvasása és egységesítése

Minden rekordhoz hozzáadjuk a forrásfájl nevét és a fájlon belüli adatrekord sorszámát. Ez lehetővé teszi, hogy egy későbbi ellenőrzés során visszakeressük az eredeti rekordot.

A migráció során az időpontot dátum-idő, a vízállást és a vízhőmérsékletet numerikus adattípussá alakítjuk. A hiányzó vízhőmérsékletű rekordokat ebben a lépésben még nem dobjuk el.


In [4]:
migrated_frames = []
source_read_log = []

for file_path in csv_files:
    raw = None
    raw, encoding, separator = read_source_csv(file_path)

    required_columns = [DATETIME_COL, WATER_LEVEL_COL, WATER_TEMP_COL]
    missing_columns = [col for col in required_columns if col not in raw.columns]

    if missing_columns:
        raise ValueError(
            f"A(z) {file_path.name} fájlból hiányzó elvárt oszlop(ok): "
            f"{missing_columns}\nA tényleges oszlopnevek: {list(raw.columns)}"
        )

    migrated = pd.DataFrame({
        "idopont": parse_datetime(raw[DATETIME_COL]),
        "vizallas_cm": parse_numeric(raw[WATER_LEVEL_COL]),
        "vizhomerseklet_C": parse_numeric(raw[WATER_TEMP_COL]),
        "forras_fajl": file_path.name,
        "forras_rekord_sorszama": np.arange(1, len(raw) + 1)
    })

    migrated_frames.append(migrated)

    source_read_log.append({
        "fájlnév": file_path.name,
        "rekordok": len(raw),
        "kódolás": encoding,
        "elválasztó": repr(separator)
    })

all_records = pd.concat(migrated_frames, ignore_index=True)

read_log_df = pd.DataFrame(source_read_log)
display(read_log_df)

print("Összes beolvasott rekord:", f"{len(all_records):,}")
display(all_records.head())


,fájlnév,rekordok,kódolás,elválasztó
0,2016-17.csv,10681,utf-8-sig,';'
1,2017-18.csv,10341,utf-8-sig,';'
2,2018-19.csv,10380,utf-8-sig,';'
3,2019-20.csv,11068,utf-8-sig,';'
4,2020-21.csv,10664,utf-8-sig,';'
5,2021-22.csv,28186,utf-8-sig,';'
6,2022-23.csv,35109,utf-8-sig,';'
7,2023-24.csv,35209,utf-8-sig,';'
8,2024-25.csv,35122,utf-8-sig,';'
9,2025-26.csv,35045,utf-8-sig,';'


Összes beolvasott rekord: 221,805


,idopont,vizallas_cm,vizhomerseklet_C,forras_fajl,forras_rekord_sorszama
0,2016-09-30 00:30:00,113.0,<NA>,2016-17.csv,1
1,2016-09-30 04:15:00,113.0,<NA>,2016-17.csv,2
2,2016-09-30 06:45:00,115.0,<NA>,2016-17.csv,3
3,2016-09-30 07:00:00,114.0,18.5,2016-17.csv,4
4,2016-09-30 07:15:00,113.0,<NA>,2016-17.csv,5


## 5. Migráció előtti adatminőségi ellenőrzés

A szűrés előtt fájlonként megvizsgáljuk, hány rekordban hiányzik vagy nem értelmezhető az időpont, a vízállás, illetve a vízhőmérséklet.

Ez a táblázat már a típuskonverzió utáni állapotot mutatja. Így az olyan szöveges vízhőmérséklet-értékek is hiányzónak számítanak, amelyeket a program nem tudott számmá alakítani.


In [5]:
quality_rows = []

for file_name, group in all_records.groupby("forras_fajl", sort=True):
    quality_rows.append({
        "fájlnév": file_name,
        "rekordok_száma": len(group),
        "hiányzó_időpont": int(group["idopont"].isna().sum()),
        "hiányzó_vízállás": int(group["vizallas_cm"].isna().sum()),
        "hiányzó_vízhőmérséklet": int(group["vizhomerseklet_C"].isna().sum())
    })

migration_quality = pd.DataFrame(quality_rows)
display(migration_quality)

print("Teljes beolvasott rekordszám:", f"{len(all_records):,}")
print("Hiányzó időpont:", f"{all_records['idopont'].isna().sum():,}")
print("Hiányzó vízállás:", f"{all_records['vizallas_cm'].isna().sum():,}")
print("Hiányzó vízhőmérséklet:", f"{all_records['vizhomerseklet_C'].isna().sum():,}")


,fájlnév,rekordok_száma,hiányzó_időpont,hiányzó_vízállás,hiányzó_vízhőmérséklet
0,2016-17.csv,10681,0,0,10381
1,2017-18.csv,10341,0,0,10337
2,2018-19.csv,10380,0,0,10378
3,2019-20.csv,11068,0,0,11067
4,2020-21.csv,10664,0,0,10391
5,2021-22.csv,28186,0,0,27822
6,2022-23.csv,35109,0,0,34749
7,2023-24.csv,35209,0,0,34845
8,2024-25.csv,35122,0,2,34757
9,2025-26.csv,35045,0,1,34683


Teljes beolvasott rekordszám: 221,805
Hiányzó időpont: 0
Hiányzó vízállás: 3
Hiányzó vízhőmérséklet: 219,410


## 6. A szűrési feltétel alkalmazása

Az első elemzési adatállományba csak azok a rekordok kerülnek, amelyeknél:
- az időpont nem hiányzó;
- a vízállás numerikus érték;
- a vízhőmérséklet numerikus érték.

A szűrés előtt és után megőrizzük a rekordszámokat, valamint fájlonként is összesítjük a megtartott rekordokat.


In [6]:
valid_datetime = all_records["idopont"].notna()
valid_water_level = all_records["vizallas_cm"].notna()
valid_water_temp = all_records["vizhomerseklet_C"].notna()

all_records["ervenyes_idopont"] = valid_datetime
all_records["ervenyes_vizallas"] = valid_water_level
all_records["ervenyes_vizhomerseklet"] = valid_water_temp

keep_mask = valid_datetime & valid_water_level & valid_water_temp

initial_filtered = all_records.loc[keep_mask].copy()

# A szűrt elemzési állományban a technikai ellenőrző oszlopokra már nincs szükség.
initial_filtered = initial_filtered.drop(
    columns=[
        "ervenyes_idopont",
        "ervenyes_vizallas",
        "ervenyes_vizhomerseklet"
    ]
)

print("Migráció előtti rekordok:", f"{len(all_records):,}")
print("Megtartott rekordok:", f"{len(initial_filtered):,}")
print("Kizárt rekordok:", f"{len(all_records) - len(initial_filtered):,}")
print(
    "Megtartott rekordok aránya:",
    f"{len(initial_filtered) / len(all_records) * 100:.2f}%"
    if len(all_records) else "n/a"
)

retained_by_file = (
    initial_filtered.groupby("forras_fajl")
    .size()
    .rename("megtartott_rekordok")
    .reset_index()
)

migration_comparison = (
    migration_quality
    .merge(retained_by_file, left_on="fájlnév", right_on="forras_fajl", how="left")
    .drop(columns=["forras_fajl"])
)
migration_comparison["megtartott_rekordok"] = (
    migration_comparison["megtartott_rekordok"].fillna(0).astype(int)
)
migration_comparison["kizárt_rekordok"] = (
    migration_comparison["rekordok_száma"]
    - migration_comparison["megtartott_rekordok"]
)

display(migration_comparison)


Migráció előtti rekordok: 221,805
Megtartott rekordok: 2,392
Kizárt rekordok: 219,413
Megtartott rekordok aránya: 1.08%


,fájlnév,rekordok_száma,hiányzó_időpont,hiányzó_vízállás,hiányzó_vízhőmérséklet,megtartott_rekordok,kizárt_rekordok
0,2016-17.csv,10681,0,0,10381,300,10381
1,2017-18.csv,10341,0,0,10337,4,10337
2,2018-19.csv,10380,0,0,10378,2,10378
3,2019-20.csv,11068,0,0,11067,1,11067
4,2020-21.csv,10664,0,0,10391,273,10391
5,2021-22.csv,28186,0,0,27822,364,27822
6,2022-23.csv,35109,0,0,34749,360,34749
7,2023-24.csv,35209,0,0,34845,364,34845
8,2024-25.csv,35122,0,2,34757,363,34759
9,2025-26.csv,35045,0,1,34683,361,34684


## 7. A szűrés eredményének ellenőrzése

A következő ellenőrzéseknek teljesülniük kell:
- a szűrt állományban nincs hiányzó időpont;
- nincs hiányzó vízállás;
- nincs hiányzó vízhőmérséklet;
- a szűrt rekordok száma nem nagyobb az eredeti rekordok számánál.

Ha bármelyik ellenőrzés sikertelen, a notebook hibát jelez.


In [7]:
assert initial_filtered["idopont"].notna().all()
assert initial_filtered["vizallas_cm"].notna().all()
assert initial_filtered["vizhomerseklet_C"].notna().all()
assert len(initial_filtered) <= len(all_records)

print("Az ellenőrzések sikeresek.")
print("A szűrt adatállományban minden rekord mindhárom változója érvényes.")


Az ellenőrzések sikeresek.
A szűrt adatállományban minden rekord mindhárom változója érvényes.


## 8. Adatállományok mentése

Két adatállományt mentünk el:

1. **Köztes, összevont állomány:** minden beolvasott rekordot tartalmaz, a típuskonverzió után, a vízhőmérséklet szerinti szűrés előtt.
2. **Első, szűrt elemzési állomány:** csak azokat a rekordokat tartalmazza, amelyeknél mindhárom vizsgált mező érvényes.

A mentés CSV és Excel formátumban is megtörténik. Az Excel-formátum kényelmes ellenőrzéshez, a CSV pedig későbbi programozott feldolgozáshoz használható.


In [8]:
interim_csv = INTERIM_DIR / "balaton_2016-2026_osszesitett_szures_elott.csv"
filtered_csv = PROCESSED_DIR / "balaton_2016-2026_vizallas_vizho_ervenyes.csv"
filtered_xlsx = PROCESSED_DIR / "balaton_2016-2026_vizallas_vizho_ervenyes.xlsx"
comparison_csv = PROCESSED_DIR / "02_migracios_osszehasonlitas_fajlonkent.csv"

all_records.to_csv(interim_csv, index=False, encoding="utf-8-sig", sep=";")
initial_filtered.to_csv(filtered_csv, index=False, encoding="utf-8-sig", sep=";")
initial_filtered.to_excel(filtered_xlsx, index=False)
migration_comparison.to_csv(
    comparison_csv, index=False, encoding="utf-8-sig", sep=";"
)

print("Mentett fájlok:")
print("Köztes, szűrés előtti CSV:", interim_csv)
print("Szűrt CSV:", filtered_csv)
print("Szűrt Excel:", filtered_xlsx)
print("Fájlonkénti összehasonlítás:", comparison_csv)


Mentett fájlok:
Köztes, szűrés előtti CSV: C:\Users\Zoli\Adatelemzéseim\Balatoni_vízállás_2016-2026_GIT\data\interim\balaton_2016-2026_osszesitett_szures_elott.csv
Szűrt CSV: C:\Users\Zoli\Adatelemzéseim\Balatoni_vízállás_2016-2026_GIT\data\processed\balaton_2016-2026_vizallas_vizho_ervenyes.csv
Szűrt Excel: C:\Users\Zoli\Adatelemzéseim\Balatoni_vízállás_2016-2026_GIT\data\processed\balaton_2016-2026_vizallas_vizho_ervenyes.xlsx
Fájlonkénti összehasonlítás: C:\Users\Zoli\Adatelemzéseim\Balatoni_vízállás_2016-2026_GIT\data\processed\02_migracios_osszehasonlitas_fajlonkent.csv


## 9. A migráció eredményének összefoglalása

A notebook végén az alábbi számokat rögzítjük:
- beolvasott CSV-fájlok száma;
- összes beolvasott rekord;
- szűrés után megtartott rekord;
- kizárt rekord;
- megtartott rekordok aránya.

A fájlonkénti összehasonlítás alapján láthatóvá válik, hogy az eredeti, vízhőmérsékletet is megkövetelő szűrés milyen mértékben csökkenti az egyes évek adatmennyiségét.

Ez a migráció még nem a teljes vízállás-adatállományt állítja elő. A következő EDA az itt létrehozott, vízhőmérsékletet is tartalmazó szűrt állományra épül, és később ennek korlátait értékeljük újra.


In [9]:
migration_summary = pd.DataFrame([{
    "forrásfájlok_száma": len(csv_files),
    "összes_beolvasott_rekord": len(all_records),
    "megtartott_rekord": len(initial_filtered),
    "kizárt_rekord": len(all_records) - len(initial_filtered),
    "megtartott_arány_%": round(
        len(initial_filtered) / len(all_records) * 100, 2
    ) if len(all_records) else np.nan
}])

display(migration_summary)


,forrásfájlok_száma,összes_beolvasott_rekord,megtartott_rekord,kizárt_rekord,megtartott_arány_%
0,10,221805,2392,219413,1.08
